# C — BVNR DOM Introspection

This notebook shows how to use the `bovnar.analytics` helpers to
inspect, assert, and validate BVNR documents interactively — useful
during format design, embedded-config review, and CI smoke-testing.

**Requires:** `libbvnr_shared.so` on `LIBBOVNAR_PATH`.

## Contents
1. [Build a test document](#1.-Build-a-test-document)
2. [DOM summary table](#2.-DOM-summary-table)
3. [Targeted node assertions](#3.-Targeted-node-assertions)
4. [Schema validation](#4.-Schema-validation)
5. [Nested struct traversal](#5.-Nested-struct-traversal)
6. [Using assert_node inside pytest](#6.-Using-assert_node-inside-pytest)


## Setup


In [ ]:
import bovnar
from bovnar.analytics import (
    assert_node,
    NodeAssertionError,
    dom_summary,
    check_schema,
)
from bovnar.writer import Writer
from bovnar.enums import BaseUnit, SIPrefix


## 1. Build a test document

Produce a `.bvnr` document that represents a sensor-node configuration.
In a real project this would come from the embedded device, a file on
disk, or a CI artifact.


In [ ]:
with Writer.to_mem() as w:
    w.write_uint("node_id",    7,     width=8)
    w.write_float("frequency", 868.1, width=64,
                  unit_si_base=BaseUnit.HERTZ,
                  unit_si_prefix=SIPrefix.MEGA)
    w.write_sint("tx_power",   14,    width=8)
    w.write_uint("interval",   60,    width=32,
                 unit_si_base=BaseUnit.SECOND)
    w.write_float("voltage",   3.3,   width=32,
                  unit_si_base=BaseUnit.VOLT)
    w.write_string("firmware",        "2.4.1")

    w.begin_struct("calibration")
    w.write_float("gain",    1.002, width=64)
    w.write_float("offset", -0.25, width=64,
                  unit_si_base=BaseUnit.KELVIN)
    w.end_struct()

bvnr_bytes = w.get_output()

doc = bovnar.dom_parse(bvnr_bytes)
print('Parse succeeded, top-level keys:', [k for k, _ in doc])


## 2. DOM summary table

`dom_summary` returns a tidy DataFrame of every top-level key: its
type, unit string, SI value, and native Python value.  Non-numeric
nodes get `NaN` for `value_si`.

This is the quickest way to check whether a freshly parsed document
looks sane before diving into individual fields.


In [ ]:
df = dom_summary(doc)
df


In [ ]:
# Quick check: all expected keys are present
expected = {"node_id", "frequency", "tx_power", "interval",
            "voltage", "firmware", "calibration"}
actual   = set(df["key"])
missing  = expected - actual
extra    = actual   - expected

print(f"Missing keys : {missing or None}")
print(f"Extra keys   : {extra   or None}")
assert not missing, f"Missing keys: {missing}"


## 3. Targeted node assertions

`assert_node` is the surgical tool: check a single node's type, unit,
and SI value in one call.  When a check fails it raises
`NodeAssertionError` (a subclass of `AssertionError`) with a
descriptive message including the `path` you provide.


In [ ]:
# Frequency: 868.1 MHz → 868 100 000 Hz in SI
assert_node(doc['frequency'],
            expected_type='float',
            expected_unit='M-Hz',
            expected_si=868_100_000.0,
            tol=1.0,
            path='frequency')
print('frequency ✓')

# Voltage: 3.3 V
assert_node(doc['voltage'],
            expected_type='float',
            expected_unit='V',
            expected_si=3.3,
            tol=1e-5,
            path='voltage')
print('voltage ✓')

# node_id: integer, no unit
assert_node(doc['node_id'],
            expected_type='int',
            path='node_id')
print('node_id ✓')


In [ ]:
# Deliberately trigger a failure to see the error message
try:
    assert_node(doc['frequency'],
                expected_unit='Hz',  # wrong — should be 'M-Hz'
                path='frequency')
except NodeAssertionError as exc:
    print(f'Caught expected error: {exc}')


## 4. Schema validation

`check_schema` validates a whole document against a declarative rule
dict in one call.  It returns a list of error strings (empty = valid).
This is useful in CI pipelines, integration tests, and data review
sessions where you want a quick pass/fail against a specification.


In [ ]:
schema = {
    'node_id':   {'type': 'INT',   'min_si': 0.0, 'max_si': 255.0},
    'frequency': {'type': 'FLOAT', 'unit_str': 'M-Hz',
                  'min_si': 863e6, 'max_si': 870e6},
    'voltage':   {'type': 'FLOAT', 'unit_str': 'V',
                  'min_si': 1.8,   'max_si': 5.5},
    'firmware':  {'type': 'STRING'},
    'humidity':  {'required': False},   # optional — absence is fine
}

errors = check_schema(doc, schema)

if errors:
    print("Schema violations:")
    for e in errors:
        print(f'  • {e}')
else:
    print('Document satisfies all schema rules ✓')


In [ ]:
# Tighten the voltage range to force a violation
schema_strict = {
    'voltage': {'type': 'FLOAT', 'max_si': 3.0},
}

errs = check_schema(doc, schema_strict)
print("Strict schema errors:", errs)


## 5. Nested struct traversal

The DOM API supports dotted path lookup via `doc.lookup(path)`.
Combine it with `assert_node` for targeted checks deep in the tree.


In [ ]:
cal_gain   = doc.lookup("calibration.gain")
cal_offset = doc.lookup("calibration.offset")

if cal_gain is None:
    raise KeyError("calibration.gain not found")
if cal_offset is None:
    raise KeyError("calibration.offset not found")

assert_node(cal_gain,   expected_type='float', path='calibration.gain')
assert_node(cal_offset, expected_type='float',
            expected_unit="K",
            path='calibration.offset')

print(f'calibration.gain   = {cal_gain.as_float()}')
print(f'calibration.offset = {cal_offset.as_float()} (unit: {cal_offset.unit_str})')


In [ ]:
# Iterate struct children
cal_node = doc['calibration']
print('calibration struct children:')
for key, child in cal_node:
    dt = child.dom_type.name
    if dt in ('INT', 'FLOAT'):
        print(f'  .{key:<10} {dt:<8} {child.value_in_base_units():.4f} {child.unit_str}')
    else:
        print(f'  .{key:<10} {dt}')


## 6. Using assert_node inside pytest

Because `NodeAssertionError` is a subclass of `AssertionError`, pytest
catches it naturally.  The pattern below shows how to share document
parsing as a fixture and write parametrised field checks.


In [ ]:
# ── This cell is documentation, not executable in the notebook. ────────
# Copy it into python/tests/test_device_config.py.

"""
import pytest
import bovnar
from bovnar.analytics import assert_node, check_schema
from conftest import needs_lib

_CONFIG_BVNR = open("device.bvnr", "rb").read()

@pytest.fixture(scope="module")
def doc():
    return bovnar.dom_parse(_CONFIG_BVNR)

@pytest.mark.parametrize("key,expected_type,expected_unit", [
    ("node_id",   "int",   ""),
    ("frequency", "float", "M-Hz"),
    ("voltage",   "float", "V"),
])
@needs_lib
def test_field_types(doc, key, expected_type, expected_unit):
    assert_node(doc[key], expected_type=expected_type,
                expected_unit=expected_unit or None, path=key)

@needs_lib
def test_schema(doc):
    schema = {
        "node_id":   {"type": "INT",   "min_si": 0, "max_si": 255},
        "frequency": {"type": "FLOAT", "unit_str": "M-Hz"},
        "firmware":  {"type": "STRING"},
    }
    errors = check_schema(doc, schema)
    assert errors == [], f"Schema errors: {errors}"
"""
